# Assignment — Bloc 5b, Sessió 9
### Síntesi FM i envolvents amb estat

**Com funciona:** Aquest notebook és la teva còpia personal, generada des de la teva tasca del Classroom. Els canvis es desen automàticament al teu Drive mentre treballes. Quan acabis, torna a la tasca del Classroom i prem **Lliura**. No cal fer «Desa una còpia»: ja estàs treballant sobre la teva.

**Per què a Colab i (al final) a Thonny?** Les parts 1 a 5 generen so com a arrays: les volem escoltar, mirar-ne l'espectre i inspeccionar els objectes cel·la a cel·la → Colab. Al final, la teva veu FM la tocaràs **en temps real** dins un callback → Thonny (`assignment.py`). Criteri de sempre: *si cal temps real, Thonny; si no, Colab.*

**Què faràs:**
1. `process(n)`: l'oscil·lador parla en mostres, com el callback. I un vibrato.
2. La "FM" per blocs: una sorpresa, i un detectiu.
3. El canvi estructural: freqüència per a cada mostra → **FM de debò**.
4. `Envelope` amb estat: ja feta, però l'has d'entendre per dins.
5. `VeuFM`: el teu instrument, fet amb les teves peces.

**Com són els enunciats d'avui:** et diuen **què** ha de fer cada peça i quina signatura té, no com escriure-la. Les classes de la S08 i el que has fet a cada part són el teu material. No esborris les cel·les d'autotest.

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sample_rate = 44100


def note_to_freq(note_number):
    """Número MIDI → freqüència (Hz). Ja la coneixes del Bloc 4."""
    return 440.0 * (2 ** ((note_number - 69) / 12))


def espectre(so, fmax=2000, titol="Espectre"):
    """Dibuixa l'espectre (en dB) d'un so, fins a fmax Hz."""
    spec = np.abs(np.fft.rfft(so * np.hanning(len(so))))
    spec_db = 20 * np.log10(spec / np.max(spec) + 1e-12)
    freqs = np.fft.rfftfreq(len(so), 1 / sample_rate)
    plt.figure(figsize=(10, 3))
    plt.plot(freqs, spec_db)
    plt.xlim(0, fmax)
    plt.ylim(-80, 3)
    plt.xlabel("freqüència (Hz)")
    plt.ylabel("dB")
    plt.title(titol)
    plt.grid(alpha=0.3)
    plt.show()

print("A punt")

## Part 1 — L'oscil·lador parla en mostres

Aquest és l'`Oscillator` de la S08, tal com el vas deixar. Té un problema per al temps real: el callback de `sounddevice` et demana **`frames` mostres** (512, per exemple), i `generate` parla en **segons**.

**TODO 1a:** afegeix un mètode `process(n_samples)` que retorni exactament `n_samples` mostres, continuant la fase igual que `generate`.

**TODO 1b:** quan acabis, `generate` i `process` tindran el mateix `if/elif` de formes d'ona repetit. Dues còpies del mateix codi són dos llocs on arreglar el mateix error. **Treu-lo a un mètode propi, `_ona(fases)`**, que rebi un array de fases i retorni l'ona; fes que els altres mètodes el cridin. (El guió baix: és d'ús intern de la classe, com `_phase`.)

Pregunta per pensar mentre ho fas: ara `generate` podria ser **una sola línia**?

Quan editis la classe, **torna a executar la cel·la** abans de l'autotest.

In [ ]:
class Oscillator:
    def __init__(self, freq=440.0, waveform='sine', sample_rate=44100):
        self.freq = freq                # paràmetres: es poden canviar des de fora
        self.waveform = waveform
        self.sample_rate = sample_rate
        self._phase = 0.0               # estat intern: on som del cicle (0 … 2π)

    def generate(self, duration):
        n = int(round(self.sample_rate * duration))
        increment = 2 * np.pi * self.freq / self.sample_rate
        fases = self._phase + increment * np.arange(n)      # continuem on ho vam deixar

        # recordem on ha de començar la PROPERA crida (com fase_sinus a la S05)
        self._phase = (self._phase + increment * n) % (2 * np.pi)

        if self.waveform == 'sine':
            return np.sin(fases)
        elif self.waveform == 'square':
            return signal.square(fases)
        elif self.waveform == 'sawtooth':
            return signal.sawtooth(fases)
        else:
            print("Forma d'ona desconeguda:", self.waveform, "→ retorno silenci")
            return np.zeros(n)

    # TODO 1a: process(n_samples)

    # TODO 1b: _ona(fases), i fes que generate i process la facin servir

    # (Part 3: aquí hi afegiràs process_freqs)

In [ ]:
# AUTOTEST 1
osc = Oscillator(freq=440)
assert hasattr(osc, 'process'), "Falta el mètode process (TODO 1a)"
b = osc.process(512)
assert b is not None, "process() retorna None"
assert len(b) == 512, f"process(512) ha de retornar 512 mostres, en retorna {len(b)}"

# 10 blocs seguits han de ser idèntics a 5120 mostres d'un cop (la fase continua)
o1 = Oscillator(440)
blocs = []
for i in range(10):
    blocs.append(o1.process(512))
un_cop = Oscillator(440).process(5120)
assert np.allclose(np.concatenate(blocs), un_cop), \
    "Els blocs no encaixen: revisa que process() actualitza self._phase"

for forma in ['sine', 'square', 'sawtooth']:
    a = Oscillator(220, forma).process(1000)
    g = Oscillator(220, forma).generate(1000 / 44100)
    assert len(a) == 1000, f"process amb '{forma}' no retorna la longitud correcta"
    assert np.allclose(a, g), f"process i generate no donen el mateix so amb '{forma}'"

assert hasattr(osc, '_ona'), "Falta el mètode _ona (TODO 1b)"
assert np.allclose(osc._ona(np.array([0.0, np.pi / 2])), [0.0, 1.0]), \
    "_ona(fases) hauria de retornar l'ona d'aquestes fases (sinus de [0, π/2] → [0, 1])"

print("✅ Part 1a/1b correcta!")

### 🔊 Prova-ho i escolta

In [ ]:
# El que farà el callback: blocs de 512 mostres, un rere l'altre
osc = Oscillator(freq=330, waveform='sawtooth')
blocs = []
for i in range(150):
    blocs.append(osc.process(512))
print("150 blocs de 512 mostres:")
display(Audio(np.concatenate(blocs), rate=sample_rate))

# 👉 Canvia osc.freq a meitat del bucle (per exemple, quan i == 75).
#    Hi ha clic en el canvi de nota? Per què no?

### Part 1c — Vibrato

Un **vibrato** és la mateixa nota, ondulant suaument d'afinació. Fem-lo amb dos `Oscillator`:
- un **LFO** lent (pocs Hz) que no sentim directament,
- un **portador** a `base_freq`, que és el que sentim.

**TODO 1c:** escriu `vibrato(base_freq, lfo_freq, depth, n_blocs, block=512)`, que retorni `n_blocs × block` mostres. Funciona **per blocs, com un callback**: a cada bloc, la freqüència del portador s'actualitza **un cop**, a `base_freq + depth × (un valor del LFO per a aquell bloc)`. Quin valor del bloc fas servir (el primer, l'últim, el del mig…) és decisió teva: per a un LFO lent gairebé no se sent la diferència. `depth` és l'excursió en Hz.

Recorda: els dos oscil·ladors es creen **un cop**, abans del bucle. (Per què?)

In [ ]:
# TODO 1c: la funció vibrato

In [ ]:
# AUTOTEST 2
v = vibrato(440, 5, 6, 40)
assert v is not None, "vibrato() retorna None"
assert len(v) == 40 * 512, f"40 blocs de 512 haurien de ser {40 * 512} mostres, n'hi ha {len(v)}"
assert np.max(np.abs(v)) <= 1.0 + 1e-6, "La sortida surt de [-1, 1]"

# Amb depth = 0 ha de ser exactament un sinus de 440 Hz
assert np.allclose(vibrato(440, 5, 0, 20), Oscillator(440).process(20 * 512)), \
    "Amb depth=0 hauria de sonar un sinus pur de base_freq"

# La freqüència es calcula com diu l'enunciat: un sol valor del LFO per bloc
# (reconstrucció independent; s'accepta qualsevol mostra del bloc, o la seva mitjana)
def vibrato_esperat(llegeix):
    """llegeix: índex de la mostra del bloc (0…511) o 'mitjana'."""
    mostres = np.arange(40 * 512)
    lfo = np.sin(2 * np.pi * 5 * mostres / 44100).reshape(40, 512)
    if llegeix == 'mitjana':
        valor = lfo.mean(axis=1)
    else:
        valor = lfo[:, llegeix]
    inc = 2 * np.pi * np.repeat(440 + 6 * valor, 512) / 44100
    return np.sin(np.concatenate([[0.0], np.cumsum(inc)[:-1]]))

coincideix = np.allclose(v, vibrato_esperat('mitjana'), atol=1e-6)
for j in range(512):
    if np.allclose(v, vibrato_esperat(j), atol=1e-6):
        coincideix = True
        break
assert coincideix, \
    "El vibrato no coincideix: un sol valor del LFO per bloc, i el portador a base_freq + depth × aquest valor"

print("✅ Part 1c correcta!")

### 🔊 Prova-ho i escolta

In [ ]:
for d in [0, 6, 20]:
    print("depth =", d, "Hz")
    display(Audio(vibrato(440, 5.5, d, 170), rate=sample_rate))

# 👉 Busca el vibrato que et sembli més "musical" (cantant, violí...) canviant
#    lfo_freq i depth. A partir de quin depth deixa de semblar vibrato?

## Part 2 — La "FM" per blocs

La **síntesi FM** (Chowning, 1973; el Yamaha DX7, 1983) fa exactament el mateix que el teu vibrato, però amb el modulador a **freqüència d'àudio**. Per exemple: portador a 440 Hz, modulador a **80 Hz**, excursió de **200 Hz**.

**✋ Abans d'executar la cel·la següent:** què sentiràs? Escriu la teva predicció aquí sota (una frase).

In [ ]:
la_meva_prediccio = """ """

so_blocs = vibrato(440, 80, 200, 170)       # la mateixa funció, només canvien els números
display(Audio(so_blocs, rate=sample_rate))
espectre(so_blocs, titol="'FM' per blocs: 440 Hz modulat per 80 Hz")

Això no és un timbre nou: és una **sirena** lenta. 🤔 Agafa el **full en paper**: hi ha un sinus de 80 Hz dibuixat i una línia vertical a cada bloc de 512 mostres. Marca on cada línia talla el sinus i uneix els punts. Què dibuixen?

### 🕵️ Detectiu

Ara que saps què passa, fes-ho servir. Experimenta amb `vibrato(440, lfo_freq, 200, 170)` a la cel·la de sota (escolta i mira l'espectre) i omple les tres respostes:

1. **`freq_congelada`**: amb quina `lfo_freq` (al voltant de 80-90 Hz) la sirena **s'atura del tot**: una nota fixa, sense moviment? Raona-ho primer al paper; després comprova-ho.
2. **`sirena_70`**: si el modulador fa 70 Hz, a quants Hz (aprox.) creus que ondularà la sirena? Calcula-ho i després escolta per confirmar.
3. **`resposta_172`**: què sents amb el **doble** de `freq_congelada`? Per què?

In [ ]:
# El teu laboratori: canvia lfo_freq i escolta
prova = vibrato(440, 80, 200, 170)
display(Audio(prova, rate=sample_rate))

freq_congelada = 0      # <-- la teva resposta (Hz)
sirena_70 = 0           # <-- la teva resposta (Hz)
resposta_172 = """ """

In [ ]:
# AUTOTEST 3
assert abs(freq_congelada - 44100 / 512) < 0.5, \
    "freq_congelada no és correcta: pensa quants blocs per segon demana el callback"
assert abs(sirena_70 - (44100 / 512 - 70)) < 1.5, \
    "sirena_70 no és correcta: compara 70 Hz amb el ritme al qual 'mirem' el modulador"
assert len(resposta_172.strip()) > 10, "Escriu què sents a 172 Hz i per què"
print("✅ Part 2 correcta!")

## Part 3 — El canvi estructural: una freqüència per a cada mostra

El problema no és la idea, és **quan** llegim el modulador: un cop per bloc (86 cops per segon) és massa poc per seguir un senyal de 80 Hz. Per fer FM de debò, el portador ha de canviar de freqüència **a cada mostra**.

**Índex de modulació (notació de Chowning, la del DX7):** en lloc de donar l'excursió en Hz, es dona `I`, sense unitats:

  **excursió (Hz) = I × freqüència del modulador**

Així, 200 Hz d'excursió amb un modulador de 80 Hz és **I = 2,5**. Quan `I` creix, apareixen més parcials.

**TODO 3a:** afegeix a la classe `Oscillator` (cel·la de la Part 1) un mètode `process_freqs(freqs)`:
- rep un **array de freqüències**, una per a cada mostra;
- retorna **tantes mostres com freqüències**;
- per a cada mostra: la fase avança segons la freqüència d'**aquella** mostra (la idea de l'increment de fase ja la tens a `process`);
- la conversió de fases a ona ja la tens feta: `_ona`. (Ara veus per què valia la pena la Part 1b?)

Torna a executar la cel·la de la classe quan acabis.

**TODO 3b:** escriu `fm_synth(carrier_freq, mod_freq, I, n_blocs, block=512)`: igual que el vibrato, però el portador rep a cada bloc **un array** de freqüències (portador + excursió × el bloc sencer del modulador).

In [ ]:
# TODO 3b: la funció fm_synth

In [ ]:
# AUTOTEST 4
osc = Oscillator(330, 'sawtooth')
assert hasattr(osc, 'process_freqs'), "Falta process_freqs a Oscillator (TODO 3a) — has tornat a executar la classe?"
r = osc.process_freqs(np.full(700, 330.0))
assert r is not None and len(r) == 700, "process_freqs ha de retornar una mostra per cada freqüència"

# Amb una freqüència constant ha de sonar exactament com process
o1 = Oscillator(330, 'sawtooth')
o2 = Oscillator(330, 'sawtooth')
a = np.concatenate([o1.process(512), o1.process(512)])
b = np.concatenate([o2.process_freqs(np.full(512, 330.0)), o2.process_freqs(np.full(512, 330.0))])
assert np.allclose(a, b, atol=1e-6), \
    "Amb freqüència constant, process_freqs hauria de donar el mateix que process (i continuar la fase)"

# FM de debò: parcials a 440 ± k·80 Hz
so = fm_synth(440, 80, 2.5, 172)
assert len(so) == 172 * 512, "fm_synth no retorna n_blocs × block mostres"
spec = np.abs(np.fft.rfft(so))
freqs = np.fft.rfftfreq(len(so), 1 / 44100)
def nivell(f):
    return np.max(spec[np.abs(freqs - f) < 3])
for f in [280, 360, 520, 600]:
    assert nivell(f) > 0.2 * np.max(spec), f"La banda lateral de {f} Hz és massa feble o no hi és: revisa la fórmula de l'excursió (I × mod_freq)"
assert nivell(400) < 0.05 * nivell(360), "Hi ha energia entre bandes laterals: el portador encara canvia per blocs?"
print("✅ Part 3 correcta!")

### 🔊 Prova-ho i escolta: abans i després

In [ ]:
abans = vibrato(440, 80, 200, 170)              # per blocs
despres = fm_synth(440, 80, 2.5, 170)           # per mostra (mateixos números: 2,5 × 80 = 200 Hz)

print("ABANS (per blocs):")
display(Audio(abans, rate=sample_rate))
espectre(abans, titol="Per blocs")
print("DESPRÉS (per mostra):")
display(Audio(despres, rate=sample_rate))
espectre(despres, titol="Per mostra: bandes laterals a 440 ± k·80 Hz")

# 👉 Prova I = 0.3, 1, 2.5, 6. Què passa amb el nombre de parcials?
# 👉 Posa mod_freq = 440 (el mateix que el portador). I 880? I 620?
#    Quan sona "afinat" i quan sona com una campana o un metall?

In [ ]:
reflexio_3 = """
Per què ara sí? Què ha canviat a l'estructura del codi, i què ha calgut tocar
(i què NO ha calgut tocar) gràcies a _ona?
"""

## Part 4 — Una envolvent que recorda on és

A la S08, `Envelope.generate(note_duration)` necessitava saber **per endavant** quant duraria la nota. En temps real no ho sabem: el `note_off` arriba quan arriba. Aquesta `Envelope` és diferent: no calcula la corba sencera, sinó que **avança mostra a mostra** i recorda on és amb dos atributs d'estat:

- `self.stage`: en quina fase és: `'idle'`, `'attack'`, `'decay'`, `'sustain'` o `'release'`
- `self.level`: el nivell actual (0 … 1)

`note_on()` i `note_off()` **només canvien l'estat**; és `process(n)` qui fa avançar la corba. Llegeix-la amb calma: l'`if/elif` de `process` és el cor de la classe. No cal completar-hi res.

In [ ]:
class Envelope:
    def __init__(self, attack=0.01, decay=0.1, sustain=0.7, release=0.2, sample_rate=44100):
        self.attack = attack            # segons
        self.decay = decay              # segons
        self.sustain = sustain          # NIVELL (0-1), no un temps
        self.release = release          # segons
        self.sample_rate = sample_rate

        self.stage = 'idle'             # estat: en quina fase som
        self.level = 0.0                # estat: nivell actual
        self._pas = 0.0                 # quant puja o baixa el nivell a cada mostra

    def note_on(self):
        self.stage = 'attack'           # comença a pujar des del nivell on sigui
        n_attack = max(1, int(self.attack * self.sample_rate))
        self._pas = 1.0 / n_attack

    def note_off(self):
        if self.stage == 'idle':
            return                      # no sonava: no hi ha res a apagar
        self.stage = 'release'          # baixa des del nivell ACTUAL fins a 0
        n_release = max(1, int(self.release * self.sample_rate))
        self._pas = self.level / n_release

    def process(self, n_samples):
        out = np.zeros(n_samples)
        for k in range(n_samples):
            if self.stage == 'attack':
                self.level = self.level + self._pas
                if self.level >= 1.0:               # hem arribat a dalt → decay
                    self.level = 1.0
                    self.stage = 'decay'
                    n_decay = max(1, int(self.decay * self.sample_rate))
                    self._pas = (1.0 - self.sustain) / n_decay
            elif self.stage == 'decay':
                self.level = self.level - self._pas
                if self.level <= self.sustain:      # hem arribat al sustain
                    self.level = self.sustain
                    self.stage = 'sustain'
            elif self.stage == 'sustain':
                self.level = self.sustain           # s'hi queda fins al note_off
            elif self.stage == 'release':
                self.level = self.level - self._pas
                if self.level <= 0.0:               # silenci → idle
                    self.level = 0.0
                    self.stage = 'idle'
            out[k] = self.level
        return out

### ✋ Prediu abans d'executar

Amb `Envelope(attack=0.01, decay=0.1, sustain=0.7, release=0.2)`, fent `note_on()` i després `process(512)` bloc a bloc:
- a quin **bloc** (0, 1, 2…) passarà de `'attack'` a `'decay'`?
- a quin bloc arribarà a `'sustain'`?

Escriu les prediccions i executa: la cel·la imprimeix l'estat al final de cada bloc.

In [ ]:
prediccio_decay = None      # <-- número de bloc
prediccio_sustain = None    # <-- número de bloc

env = Envelope(attack=0.01, decay=0.1, sustain=0.7, release=0.2)
env.note_on()
for b in range(14):
    env.process(512)
    print("bloc", b, " stage:", env.stage, "  level:", round(env.level, 3))
print("Les teves prediccions:", prediccio_decay, prediccio_sustain)

Dues situacions que només passen en temps real. **Prediu primer** i després executa:

1. Deixes anar la tecla **durant l'attack** (la nota encara no ha arribat a dalt). El so salta a zero, o baixa des d'on era?
2. Tornes a prémer la tecla **durant el release**. Torna a començar des de zero (clic!) o puja des d'on era?

In [ ]:
env = Envelope(attack=0.3, decay=0.1, sustain=0.7, release=0.2)
corbes = []
env.note_on()
for b in range(8):                  # ~0,09 s: encara som a l'attack
    corbes.append(env.process(512))
print("abans del note_off:", env.stage, round(env.level, 3))
env.note_off()
for b in range(10):
    corbes.append(env.process(512))
env.note_on()                       # nova nota a mig release
for b in range(30):
    corbes.append(env.process(512))
env.note_off()
for b in range(25):
    corbes.append(env.process(512))

corba = np.concatenate(corbes)
plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(corba)) / sample_rate, corba)
plt.title("note_off a l'attack, i note_on a mig release")
plt.xlabel("temps (s)")
plt.show()
display(Audio(Oscillator(220, 'sawtooth').process(len(corba)) * corba, rate=sample_rate))

# 👉 Busca a process() la línia exacta que fa que no hi hagi salt en cada cas.

## Part 5 — `VeuFM`: el teu instrument

### ✏️ Primer, en paper (3')

A la S08 vau dissenyar una groovebox. Avui el nivell és més petit: **una veu** (una nota alhora), no un sintetitzador sencer. Abans d'escriure, respon al paper:
- Quines classes **ja tens** que pots fer servir a dins? Quantes de cada?
- Què és **configuració** (es tria un cop, a `__init__`) i què és **estat** (canvia a cada nota)?

### Especificació

| | |
|---|---|
| Constructor | `VeuFM(ratio=1.0, I_max=3.0, attack=0.01, decay=0.2, sustain=0.6, release=0.3)` |
| A dins | un portador, un modulador i una envolvent, **creats un sol cop** (`self.carrier`, `self.modulator`, `self.env`) |
| `note_on(note_number, velocity=100)` | el portador a la freqüència de la nota; el modulador a **portador × `ratio`**; guarda el guany de la velocity (`velocity / 127`); dispara l'envolvent |
| `note_off()` | inicia el release |
| `process(n_samples)` | retorna `n_samples` mostres de FM, multiplicades per la corba de l'envolvent i pel guany |

**El detall que fa sonar un instrument (Chowning):** l'índex no és fix. A cada mostra, **`I = I_max × corba de l'envolvent`**: la nota és brillant a l'atac i s'enfosqueix quan s'apaga, com un metall o una campana reals.

**TODO 5:** escriu la classe `VeuFM`.

In [ ]:
# TODO 5: la classe VeuFM

Per escoltar-la sense callback, aquesta funció fa de "teclat": per a cada nota fa `note_on`, demana blocs amb `process`, i fa `note_off` quan toca.

In [ ]:
def toca(veu, notes, block=512):
    """notes: llista de (nota_midi, velocity, segons_premuda, segons_total)."""
    trossos = []
    for nota, vel, premuda, total in notes:
        veu.note_on(nota, vel)
        blocs_premuda = int(premuda * sample_rate / block)
        blocs_total = int(total * sample_rate / block)
        for b in range(blocs_total):
            if b == blocs_premuda:
                veu.note_off()
            trossos.append(veu.process(block))
    return np.concatenate(trossos)

In [ ]:
# AUTOTEST 5
veu = VeuFM(ratio=2.0, I_max=4.0)
carrier_inicial = veu.carrier
modulator_inicial = veu.modulator

veu.note_on(57, 127)                                    # La3 = 220 Hz
assert abs(veu.carrier.freq - 220.0) < 0.01, "La freqüència del portador no correspon a la nota"
assert abs(veu.modulator.freq - 440.0) < 0.01, "El modulador hauria de ser portador × ratio"
bloc = veu.process(512)
assert bloc is not None and len(bloc) == 512, "process(512) ha de retornar 512 mostres"

so = toca(veu, [(60, 127, 0.5, 1.0), (64, 127, 0.5, 1.0)])
assert veu.carrier is carrier_inicial and veu.modulator is modulator_inicial, \
    "Els oscil·ladors s'han de crear UN sol cop, a __init__ (no a cada nota)"
assert np.max(np.abs(so)) <= 1.0 + 1e-6, "La sortida surt de [-1, 1]"
assert np.max(np.abs(so[-2000:])) < 1e-6, "Després del release hi hauria d'haver silenci"

fort = toca(VeuFM(), [(60, 127, 0.3, 0.6)])
fluix = toca(VeuFM(), [(60, 64, 0.3, 0.6)])
assert abs(np.max(np.abs(fluix)) / np.max(np.abs(fort)) - 64 / 127) < 0.03, "La velocity no s'aplica com a guany"

def amplada(x):
    s = np.abs(np.fft.rfft(x))
    d = 20 * np.log10(s / np.max(s) + 1e-12)
    f = np.fft.rfftfreq(len(x), 1 / 44100)
    return f[d > -20].max() - f[d > -20].min()
fosc = toca(VeuFM(ratio=1, I_max=5, decay=0.6, sustain=0.2), [(57, 110, 1.0, 1.4)])
assert amplada(fosc[441:441 + 8192]) > 1.5 * amplada(fosc[35000:35000 + 8192]), \
    "La nota hauria de ser més brillant a l'atac que al final: l'índex segueix l'envolvent?"
print("✅ Part 5 correcta!")

### 🔊 Prova-ho i escolta: el mapa de timbres

La cel·la toca la mateixa nota amb 4 `ratio` × 2 `I_max`. Escolta-les i **omple la taula** (doble clic aquí per editar): una paraula o dues per a cada so (fusta, campana, metall, nasal, orgue, buit…).

| `ratio` | `I_max` = 1 | `I_max` = 5 |
|---|---|---|
| 1 | | |
| 2 | | |
| 3.5 | | |
| 1.41 | | |

Dues preguntes:
- Amb quins `ratio` la nota sona **afinada** i amb quins no? Mira els espectres: on són els parcials?
- Amb `ratio = 3.5`, quina nota **sents**? És la que has tocat?

In [ ]:
for ratio in [1, 2, 3.5, 1.41]:
    for I_max in [1, 5]:
        print("ratio", ratio, "  I_max", I_max)
        so = toca(VeuFM(ratio=ratio, I_max=I_max, sustain=0.5), [(55, 100, 0.8, 1.2)])
        display(Audio(so, rate=sample_rate))
    espectre(toca(VeuFM(ratio=ratio, I_max=5, sustain=0.8), [(55, 100, 1.0, 1.2)])[4410:40000],
             fmax=3000, titol="ratio " + str(ratio) + "  (I_max = 5), nota de 196 Hz")

In [ ]:
# Dos presets clàssics. 👉 Canvia'n els paràmetres: què fa que la campana sigui una campana?
metall = VeuFM(ratio=1, I_max=5, attack=0.06, decay=0.3, sustain=0.6, release=0.2)
campana = VeuFM(ratio=1.41, I_max=6, attack=0.001, decay=2.5, sustain=0.0, release=1.0)

print("Metall (brass):")
display(Audio(toca(metall, [(55, 100, 0.4, 0.5), (59, 100, 0.4, 0.5), (62, 110, 0.9, 1.2)]), rate=sample_rate))
print("Campana:")
display(Audio(toca(campana, [(72, 100, 0.1, 2.5)]), rate=sample_rate))

## Part 6 — Munta-ho i escolta-ho

Fes una frase d'**almenys 6 notes** amb **almenys dues `VeuFM` de timbre diferent** (els teus presets, no els de dalt tal qual). Pots encadenar-les (`np.concatenate`) o fer-les sonar alhora (mateixa longitud i sumar). Si la suma passa d'1, normalitza.

In [ ]:
# TODO 6: la teva frase

la_meva_frase = np.array([])   # <-- substitueix

In [ ]:
# AUTOTEST 6
assert len(la_meva_frase) > 1.0 * sample_rate, "La frase sembla massa curta (almenys 6 notes)"
assert np.max(np.abs(la_meva_frase)) <= 1.0 + 1e-6, "La frase satura (passa d'1): normalitza-la"
print("✅ Part 6 correcta! Escolta-la a la cel·la següent.")

In [ ]:
# 🔊 La teva frase
Audio(la_meva_frase, rate=sample_rate)

---
### I ara, a Thonny

Obre `assignment.py`: hi enganxaràs el teu `Oscillator`, l'`Envelope` i la teva `VeuFM`, i la tocaràs **en temps real** dins un callback. Allà faràs que la velocity controli també la brillantor.

---
## 🚀 Challenges (opcional)

1. **Feedback (com el DX7):** un sol oscil·lador que es modula **a si mateix**: la **fase** de cada mostra es desplaça segons l'última mostra que ha generat, `sortida = sin(fase + feedback × sortida_anterior)`. Això és PM, com al DX7 (mira la cheat sheet: FM o PM?). Comença amb un factor de feedback petit i puja'l. Què passa a partir de cert valor? (Pista: no ho podràs fer amb `process` vectoritzat: necessites el bucle mostra a mostra.)
2. **Tres operadors en pila:** un modulador que modula un modulador que modula el portador. Quants paràmetres nous apareixen? Com faries una classe que en fos una peça reutilitzable?
3. **La campana perfecta:** a les campanes reals, els parcials aguts s'apaguen abans que els greus. Prova de fer que l'índex tingui la **seva pròpia** envolvent (més curta que la del volum).
4. **Mesura't:** cronometra `process` i `process_freqs` amb `time.perf_counter()` generant 10 segons. Quantes vegades és més lent el bucle mostra a mostra? (Ho reprendrem a la Sessió 10.)

In [ ]:
# El teu codi dels challenges aquí